In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [ ]:
import torch
import numpy as np
import random

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [ ]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [ ]:
# model_type = 'llama'
# MODEL_VERSION='3.1'
# MODEL_SIZE='8B'

# model_type = 'gemma'
# MODEL_VERSION='2'
# # MODEL_VERSION='3'
# # MODEL_SIZE='1B'
# MODEL_SIZE='9B'
# # MODEL_SIZE='12B'

model_type = 'qwen'
MODEL_VERSION='2.5'
# MODEL_VERSION='3'
# MODEL_SIZE='4B'
# MODEL_SIZE='8B'
MODEL_SIZE='14B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
hidden_layers = list(range(-llm.language_model.config.num_hidden_layers+1, 0))
print(hidden_layers)

[-31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1]


Training

In [7]:
# categories = ["physical", "texture", "time", "complexity", "logic", "state", "social"]
categories = ["social"]
antonym_pairs = []

for category in categories:
    with open(f"../data/adjectives/{category}_antonym_pairs.pkl", 'rb') as file:
        antonyms = pickle.load(file)
    
    for p in antonyms:
        antonym_pairs.append(tuple([p[0].lower(), p[1].lower(), category]))

In [8]:
print(antonym_pairs[:5])

[('honest', 'dishonest', 'social'), ('polite', 'rude', 'social'), ('generous', 'stingy', 'social'), ('brave', 'cowardly', 'social'), ('innocent', 'guilty', 'social')]


In [9]:
# test_size = 0.0
# test_size = 0.1
test_size = 0.3
# test_size = 0.5
# test_size = 0.7

print("Total data:", len(antonym_pairs))
print(antonym_pairs[:5])

if test_size == 0.0:
    train_data_t = antonym_pairs
    random.shuffle(train_data_t)
    test_data = []
else:
    train_data_t, test_data = train_test_split(antonym_pairs, test_size=test_size, random_state=SEED)

print("Training data normal:", len(train_data_t))
print(train_data_t[:5])

swap_train_data = [(b, a, c) for a, b, c in train_data_t]
print("Training data swapped:", len(swap_train_data))
print(swap_train_data[:5])

train_data = train_data_t + swap_train_data
print("Training data:", len(train_data))
print(train_data[:5])

print("Testing data:", len(test_data))
print(test_data[:5])

# t1_list = [tuple(item.lower() for item in tpl) for tpl in train_data]
# t2_list = [tuple(item.lower() for item in tpl) for tpl in test_data]

# train_data = t1_list
# test_data = t2_list

Total data: 106
[('honest', 'dishonest', 'social'), ('polite', 'rude', 'social'), ('generous', 'stingy', 'social'), ('brave', 'cowardly', 'social'), ('innocent', 'guilty', 'social')]
Training data normal: 74
[('altruistic', 'egotistical', 'social'), ('trusting', 'suspicious', 'social'), ('philanthropic', 'miserly', 'social'), ('encouraging', 'discouraging', 'social'), ('funny', 'serious', 'social')]
Training data swapped: 74
[('egotistical', 'altruistic', 'social'), ('suspicious', 'trusting', 'social'), ('miserly', 'philanthropic', 'social'), ('discouraging', 'encouraging', 'social'), ('serious', 'funny', 'social')]
Training data: 148
[('altruistic', 'egotistical', 'social'), ('trusting', 'suspicious', 'social'), ('philanthropic', 'miserly', 'social'), ('encouraging', 'discouraging', 'social'), ('funny', 'serious', 'social')]
Testing data: 32
[('industrious', 'slothful', 'social'), ('cheerful', 'gloomy', 'social'), ('noble', 'base', 'social'), ('generous', 'stingy', 'social'), ('please

In [10]:
print(test_data)

[('industrious', 'slothful', 'social'), ('cheerful', 'gloomy', 'social'), ('noble', 'base', 'social'), ('generous', 'stingy', 'social'), ('pleased', 'annoyed', 'social'), ('diligent', 'lazy', 'social'), ('discreet', 'indiscreet', 'social'), ('joyful', 'miserable', 'social'), ('thrilled', 'indifferent', 'social'), ('magnanimous', 'petty', 'social'), ('moral', 'immoral', 'social'), ('content', 'dissatisfied', 'social'), ('loving', 'hateful', 'social'), ('agreeable', 'disagreeable', 'social'), ('just', 'unjust', 'social'), ('considerate', 'inconsiderate', 'social'), ('hopeful', 'hopeless', 'social'), ('incorruptible', 'venal', 'social'), ('benign', 'malignant', 'social'), ('attentive', 'neglectful', 'social'), ('dependable', 'undependable', 'social'), ('ecstatic', 'despondent', 'social'), ('amused', 'unamused', 'social'), ('devoted', 'unfaithful', 'social'), ('proud', 'ashamed', 'social'), ('brave', 'cowardly', 'social'), ('good', 'evil', 'social'), ('sympathetic', 'unsympathetic', 'socia

In [11]:
# path = '../all_gitignore/xRFM/test/new_class0/'
path = '../all_gitignore/xRFM/test/new_class1/'

# path = '../all_gitignore/directions_adjectives_llama/{}/'
# path = '../all_gitignore/xRFM/directions_adjectives_qwen/{}/'
# path = '../all_gitignore/xRFM/directions_adjectives_gemma2/{}/'

X_train, Y_train = read_tuples_with_category(llm, train_data, path=path)

if len(test_data) != 0:
    X_test, Y_test = read_tuples_with_category(llm, test_data, path=path)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_metho

In [12]:
# # multiple variations

# paths = [
#     '../all_gitignore/xRFM/test/new_class0/', 
#     '../all_gitignore/xRFM/test/old_class0/', 
#     '../all_gitignore/xRFM/test/new_class1/', 
#     '../all_gitignore/xRFM/test/old_class1/', 
# ]


# X_train, Y_train = read_tuples_multiple(llm, train_data, paths=paths)

# if len(test_data) != 0:
#     X_test, Y_test = read_tuples_multiple(llm, test_data, paths=paths)


In [13]:
lrr_models = LRR_auto(X_train, Y_train)
# lrr_models = LRR_auto(X_train, Y_train, alpha=None)

Running with fixed alpha: 10000.0
--------------------------------------------------
Layer -1 done.
--------------------------------------------------
Layer -2 done.
--------------------------------------------------
Layer -3 done.
--------------------------------------------------
Layer -4 done.
--------------------------------------------------
Layer -5 done.
--------------------------------------------------
Layer -6 done.
--------------------------------------------------
Layer -7 done.
--------------------------------------------------
Layer -8 done.
--------------------------------------------------
Layer -9 done.
--------------------------------------------------
Layer -10 done.
--------------------------------------------------
Layer -11 done.
--------------------------------------------------
Layer -12 done.
--------------------------------------------------
Layer -13 done.
--------------------------------------------------
Layer -14 done.
-------------------------------------

In [14]:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/social(new_class0)_({test_size}).pkl', 'wb') as file:
with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/social(new_class1)_({test_size}).pkl', 'wb') as file:

# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/social_({test_size}).pkl', 'wb') as file:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/aug_social_({test_size}).pkl', 'wb') as file:

# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_models_({test_size}).pkl', 'wb') as file:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/qwen4b/combined_models.pkl', 'wb') as file:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/gemma9b/combined_models.pkl', 'wb') as file:
    pickle.dump(lrr_models, file)

In [15]:
# All layers in one

In [16]:
print(X_train[-1].shape)
print(Y_train[-1].shape)

torch.Size([148, 4096])
torch.Size([148, 4096])


In [17]:
big_X = torch.vstack(list(X_train.values())).cpu().numpy()
big_Y = torch.vstack(list(Y_train.values())).cpu().numpy()

print(big_X.shape)
print(big_Y.shape)

(4588, 4096)
(4588, 4096)


In [18]:
from sklearn.utils import shuffle

big_X_shuffled, big_Y_shuffled = shuffle(big_X, big_Y, random_state=SEED)

In [19]:
flag = 0
# flag = 1

print_error = True
# x = X[i].cpu().numpy()
# y = Y[i].cpu().numpy()

if flag == 1:
    # alpha = 10000.0
    alpha = 1000.0
    reg_lrr = make_pipeline(StandardScaler(), Ridge(alpha=alpha, solver='cholesky'))

elif flag == 0:
    alphas = 10.0 ** np.arange(2, 6)  # log grid
    print(f"Running with alpha: {alphas}")
    reg_lrr = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=10))

model_lrr = TransformedTargetRegressor(regressor=reg_lrr, transformer=StandardScaler())

model_lrr.fit(big_X_shuffled, big_Y_shuffled)

if flag == 0:
    best_alpha_lrr = model_lrr.regressor_.named_steps["ridgecv"].alpha_
    print(f"Best lambda: {best_alpha_lrr}")

if print_error:
    y_pred = model_lrr.predict(big_X_shuffled)
    mse = mean_squared_error(big_Y_shuffled, y_pred)
    rmse = np.sqrt(mse)

    r2 = r2_score(big_Y_shuffled, y_pred)

    print(f"Training RMSE: {rmse:.4f}, Training R2: {r2:.4f}")

# xtx = x.T @ x
# A = torch.linalg.solve(xtx + lambda_reg * torch.eye(d).to("cuda"), x.T @ y) # switch to more robust

print(f"Done.")

# lrr_models[i] = model_lrr

Running with alpha: [   100.   1000.  10000. 100000.]
Best lambda: 100.0
Training RMSE: 0.0028, Training R2: 0.9563
Done.


In [20]:
lrr_models = {i: model_lrr for i in hidden_layers}

In [21]:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_all_layers_models.pkl', 'wb') as file:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_all_layers_models_({test_size}).pkl', 'wb') as file:

# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/social(new_class0)_all_layers_models_({test_size}).pkl', 'wb') as file:
with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/social(new_class1)_all_layers_models_({test_size}).pkl', 'wb') as file:
    
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/social_all_layers_models_({test_size}).pkl', 'wb') as file:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/aug_social_all_layers_models_({test_size}).pkl', 'wb') as file:
    pickle.dump(lrr_models, file)

In [ ]:
layer = -15
weight_t, bias_t = get_W_b(lrr_models[layer])
va1, ve1 = eig(weight_t)

In [ ]:
va1_mag = sorted(np.abs(va1), reverse=True)
va1_real = sorted([i.real for i in va1], reverse=True)
va1_imag = sorted([i.imag for i in va1], reverse=True)

In [ ]:
plt.figure(figsize=(8, 3))

plt.plot(va1_real)
# plt.plot(va1_imag)
# plt.plot(va1_mag)

plt.ylim(-1.5, 1.5)
# plt.xlim(0, 500)
plt.title(f"layer {layer}")
plt.xlabel("direction")
plt.ylabel("eigenvalue")
plt.grid(True)
plt.show()

Testing

In [ ]:
test_size = 0.0
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_models_({test_size}).pkl', 'rb') as file:
with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_all_layers_models_({test_size}).pkl', 'rb') as file:

# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/social_models.pkl', 'rb') as file:

# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_models.pkl', 'rb') as file:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/gemma9b/combined_models.pkl', 'rb') as file:

# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_all_layers_models.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [ ]:
# [('itchy', 'relieved', 'texture'), ('colorful', 'colorless', 'texture'), ('expansive', 'confined', 'physical'), ('fat', 'skinny', 'physical'), ('sympathetic', 'unsympathetic', 'social'), ('kind', 'cruel', 'social'), ('dynamic', 'stagnant', 'time'), ('just', 'unjust', 'social'), ('clever', 'clumsy', 'logic'), ('towering', 'squat', 'physical'), ('perpendicular', 'parallel', 'physical'), ('precise', 'inaccurate', 'complexity'), ('solid', 'hollow', 'physical'), ('rotten', 'ripe', 'texture'), ('agonizing', 'soothing', 'texture'), ('squeaky', 'deep', 'texture'), ('valid', 'void', 'state'), ('incompetent', 'competent', 'complexity'), ('friendly', 'unfriendly', 'social'), ('clean', 'dirty', 'physical'), ('continuous', 'sporadic', 'time'), ('polite', 'rude', 'social'), ('virgin', 'used', 'time'), ('scented', 'odorless', 'texture'), ('amateurish', 'professional', 'complexity'), ('knotty', 'smooth', 'complexity'), ('early', 'late', 'time'), ('developing', 'declining', 'time'), ('brave', 'cowardly', 'social'), ('right', 'wrong', 'logic'), ('huge', 'tiny', 'physical'), ('good', 'evil', 'social'), ('serene', 'agitated', 'social'), ('harmonious', 'grating', 'texture'), ('sure', 'uncertain', 'complexity'), ('tangible', 'intangible', 'logic'), ('righteous', 'wicked', 'social'), ('here', 'gone', 'state'), ('soaked', 'parched', 'physical'), ('strenuous', 'facile', 'complexity'), ('diurnal', 'nocturnal', 'time'), ('wide', 'narrow', 'physical'), ('long-term', 'short-term', 'time'), ('obligatory', 'discretionary', 'logic'), ('hungry', 'sated', 'state'), ('innocent', 'guilty', 'social'), ('deep', 'superficial', 'complexity'), ('specific', 'general', 'complexity'), ('energetic', 'weary', 'state'), ('swift', 'leisurely', 'physical'), ('safe', 'dangerous', 'state'), ('unblemished', 'marred', 'state'), ('required', 'optional', 'logic'), ('activated', 'deactivated', 'state'), ('obscured', 'revealed', 'state'), ('arduous', 'effortless', 'complexity'), ('objective', 'subjective', 'logic'), ('playful', 'somber', 'social'), ('legal', 'illegal', 'logic'), ('logical', 'illogical', 'logic'), ('fugitive', 'durable', 'time'), ('manageable', 'unwieldy', 'complexity'), ('magnanimous', 'petty', 'social'), ('troublesome', 'convenient', 'complexity'), ('applicable', 'inapplicable', 'logic'), ('recent', 'remote', 'time'), ('dead', 'alive', 'state'), ('correct', 'incorrect', 'logic'), ('definite', 'vague', 'complexity'), ('chronic', 'acute', 'time'), ('animate', 'inanimate', 'logic'), ('covered', 'uncovered', 'state'), ('confident', 'insecure', 'social'), ('content', 'dissatisfied', 'social')]

In [ ]:
# ('precise', 'inaccurate', 'complexity') ('rotten', 'ripe', 'texture') ('here', 'gone', 'state') ('manageable', 'unwieldy', 'complexity') ('covered', 'uncovered', 'state')

In [ ]:
# coef = 0.65
# coef = 0.7 # llama
coef = 0.75
# coef = 0.8
# coef = 0.9

# coef = 2.1
# coef = 2.3
# coef = 2.5 # qwen

# coef = 9.0
# coef = 10.0
# coef = 15.0
# coef = 16.0 # gemma
# coef = 18.0


max_tokens = 200

prompts = ["It seems the family next doors has fallen ill."] # ('sympathetic', 'unsympathetic', 'social')
# prompts = ["I am walking home alone at night, I am a little scared.",] # ('brave', 'cowardly', 'social')
# prompts = ["I am walking home alone at night, should I be scared?",] # ('brave', 'cowardly', 'social') - g
# prompts = ["You are designing a new vehicle. Describe its size and weight."] # ('wide', 'narrow', 'physical')
# prompts = ["Explain the triple point of water."] # ('deep', 'superficial', 'complexity')
# prompts = ["Analyze the following assertion. Assertion: You can always take the ducks from the public park, they are free."] # ('legal', 'illegal', 'logic') - works better after inversion
# prompts = ["Is the following statement correct or not?"] # ('correct', 'incorrect', 'logic') - weird!!!
# prompts = ["Describe this sound."] # ('agonizing', 'soothing', 'texture')
# prompts = ["Describe the following event or entity. Subject: ."] # ('long-term', 'short-term', 'time')
# prompts = ["I am walking home alone at night, is it safe?"] # ('safe', 'dangerous', 'state') - almost

# prompts = ["What is weight of a football?"] # ('precise', 'inaccurate', 'complexity')

c1 = "sympathetic"
c2 = "unsympathetic"

category = "social"

path = f'../all_gitignore/directions_adjectives_llama/{category}/'
# path = f'../all_gitignore/xRFM/directions_adjectives_qwen/{category}/'
# path = f'../all_gitignore/xRFM/directions_adjectives_gemma2/{category}/'

c1_controller = load_controller(llm, c1, path=path)
orig_c1 = c1_controller.directions
c2_controller = load_controller(llm, c2, path=path)
orig_c2 = c2_controller.directions



# c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# orig_c1 = c1_controller.directions
# c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# orig_c2 = c2_controller.directions

print(f"Coef: {coef}")

out = test_concept_vector(c1_controller, concept=c1, prompts=prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(c2_controller, concept=c2, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


# coef = 0.8

inv_c1_lrr = apply_auto(c1_controller.directions, lrr_models)
c1_controller.directions = inv_c1_lrr
out = test_concept_vector(c1_controller, concept=f"combined inverted {c1}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)
# out = test_concept_vector(c1_controller, concept=f"combined inverted {c1}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False, qwen=True)

inv_c2_lrr = apply_auto(c2_controller.directions, lrr_models)
c2_controller.directions = inv_c2_lrr
out = test_concept_vector(c2_controller, concept=f"combined inverted {c2}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)
# out = test_concept_vector(c2_controller, concept=f"combined inverted {c2}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False, qwen=True)

In [ ]:
coef = 0.8

inv_c1_lrr = apply_auto(orig_c1, lrr_models)
c1_controller.directions = inv_c1_lrr
out = test_concept_vector(c1_controller, concept=f"combined inverted {c1}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)
# out = test_concept_vector(c1_controller, concept=f"combined inverted {c1}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False, qwen=True)

coef = 0.8

inv_c2_lrr = apply_auto(orig_c2, lrr_models)
c2_controller.directions = inv_c2_lrr
out = test_concept_vector(c2_controller, concept=f"combined inverted {c2}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)
# out = test_concept_vector(c2_controller, concept=f"combined inverted {c2}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False, qwen=True)

Compare

In [ ]:
with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_all_layers_models.pkl', 'rb') as file:
    lrr_models_c_all = pickle.load(file)

In [ ]:
with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_models.pkl', 'rb') as file:
    lrr_models_c = pickle.load(file)

In [ ]:
category = "social"

with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/{category}_models.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [ ]:
coef = 0.75
max_tokens = 200

prompts = ["It seems the family next doors has fallen ill."] # ('sympathetic', 'unsympathetic', 'social')

# prompts = ["What is weight of a football?"] # ('precise', 'inaccurate', 'complexity')

c1 = "sympathetic"
c2 = "unsympathetic"

category = "social"

c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
orig_c1 = c1_controller.directions
c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
orig_c2 = c2_controller.directions

print(f"Coef: {coef}")

out = test_concept_vector(c1_controller, concept=c1, prompts=prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(c2_controller, concept=c2, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


inv_c1_lrr = apply_auto(orig_c1, lrr_models)
c1_controller.directions = inv_c1_lrr
out = test_concept_vector(c1_controller, concept=f"inverted {c1}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

# inv_c2_lrr = apply_auto(c2_controller.directions, lrr_models)
# c2_controller.directions = inv_c2_lrr
# out = test_concept_vector(c2_controller, concept=f"combined inverted {c2}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


inv_c1_lrr_c = apply_auto(orig_c1, lrr_models_c)
c1_controller.directions = inv_c1_lrr_c
out = test_concept_vector(c1_controller, concept=f"combined inverted {c1}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


inv_c1_lrr_c_all = apply_auto(orig_c1, lrr_models_c_all)
c1_controller.directions = inv_c1_lrr_c_all
out = test_concept_vector(c1_controller, concept=f"combined all layers inverted {c1}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


In [ ]:
compare_pearson(inv_c1_lrr, inv_c1_lrr_c)

In [ ]:
compare_pearson(inv_c1_lrr, inv_c1_lrr_c_all)

In [ ]:
compare_pearson(inv_c1_lrr_c, inv_c1_lrr_c_all)